# Imports

In [ ]:
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import Flatten
from keras.layers import BatchNormalization
from keras.layers import Conv2D
from keras.layers import MaxPooling2D
from keras.layers import GlobalAveragePooling2D
from keras.layers import Dropout
from keras.layers import Rescaling
from keras.layers import RandomFlip
from keras.layers import RandomRotation
from keras.layers import RandomZoom
from keras.layers import RandomColorJitter
from keras.optimizers import AdamW
import pathlib
import keras
import keras_tuner

import tensorflow as tf

## Download dataset

In [21]:

DATASET_URL = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
DATASET_NAME = "flower_photos"
CACHE_DIR = "."

data_dir = pathlib.Path(
    tf.keras.utils.get_file(
        origin    = DATASET_URL,
        fname     = DATASET_NAME,
        cache_dir = CACHE_DIR,
        untar     = True,
        extract   = True,
    )
)

### Parameters

In [22]:
IMG_HEIGHT = 180
IMG_WIDTH  = 180
BATCH_SIZE = 32
SEED       = 123

VAL_SPLIT  = 0.2
AUTOTUNE = tf.data.AUTOTUNE

### Util functions

In [23]:
train_ds = keras.utils.image_dataset_from_directory(
  data_dir / DATASET_NAME,
  validation_split=VAL_SPLIT,
  subset="training",
  seed=SEED,
  image_size=(IMG_HEIGHT, IMG_WIDTH),
  batch_size=BATCH_SIZE)

val_ds = keras.utils.image_dataset_from_directory(
  data_dir / DATASET_NAME,
  validation_split=VAL_SPLIT,
  subset="validation",
  seed=SEED,
  image_size=(IMG_HEIGHT, IMG_WIDTH),
  batch_size=BATCH_SIZE)


# train_ds = configure_for_performance(train_ds)
# val_ds = configure_for_performance(val_ds)

Found 3670 files belonging to 5 classes.
Using 2936 files for training.
Found 3670 files belonging to 5 classes.
Using 734 files for validation.


### Layers

In [ ]:
# Altra opzione senza random search
model = Sequential()
model.add(Rescaling(1./255))
model.add(RandomFlip("horizontal_and_vertical"))
model.add(RandomColorJitter())
model.add(RandomRotation(0.2))
model.add(RandomZoom(0.1))

model.add(Conv2D(32, 5, activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D())

model.add(Conv2D(64, 3, activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D())

model.add(Conv2D(128, 3, activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D())

model.add(Conv2D(256, 3, activation='relu'))
model.add(BatchNormalization())
model.add(GlobalAveragePooling2D())

model.add(Dense(256, activation='relu'))
model.add(Dropout(0.5))

model.add(Dense(5))

model.compile(
    optimizer=AdamW(1e-4, 1e-4),
    loss=tf.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy'])

In [25]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=10),
    keras.callbacks.ModelCheckpoint(
        filepath="best_checkpoint.keras",
        monitor='val_accuracy',
        mode='max',
        save_best_only=True)
]

model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=100,
    callbacks=callbacks
)

Epoch 1/100
92/92 ━━━━━━━━━━━━━━━━━━━━ 53s 533ms/step - accuracy: 0.5150 - loss: 1.2317 - val_accuracy: 0.2398 - val_loss: 1.9981
Epoch 2/100
92/92 ━━━━━━━━━━━━━━━━━━━━ 47s 505ms/step - accuracy: 0.5681 - loss: 1.1038 - val_accuracy: 0.2398 - val_loss: 2.4148
Epoch 3/100
92/92 ━━━━━━━━━━━━━━━━━━━━ 46s 501ms/step - accuracy: 0.6005 - loss: 1.0378 - val_accuracy: 0.2425 - val_loss: 2.0764
Epoch 4/100
92/92 ━━━━━━━━━━━━━━━━━━━━ 47s 507ms/step - accuracy: 0.6117 - loss: 1.0130 - val_accuracy: 0.2861 - val_loss: 1.6774
Epoch 5/100
92/92 ━━━━━━━━━━━━━━━━━━━━ 46s 505ms/step - accuracy: 0.6121 - loss: 0.9709 - val_accuracy: 0.4387 - val_loss: 1.2459
Epoch 6/100
92/92 ━━━━━━━━━━━━━━━━━━━━ 46s 504ms/step - accuracy: 0.6393 - loss: 0.9405 - val_accuracy: 0.5790 - val_loss: 1.0165
Epoch 7/100
92/92 ━━━━━━━━━━━━━━━━━━━━ 46s 503ms/step - accuracy: 0.6485 - loss: 0.9205 - val_accuracy: 0.6499 - val_loss: 0.8994
Epoch 8/100
92/92 ━━━━━━━━━━━━━━━━━━━━ 47s 507ms/step - accuracy: 0.6475 - loss: 0.9063 - 

In [26]:
# model.save('best_78_61%.keras')
modeltry = keras.models.load_model('best_checkpoint.keras')
print("Evaluate on test data")
results = modeltry.evaluate(val_ds)
print("test loss, test acc:", results)
# tuner = keras_tuner.RandomSearch(
#     build_model,
#     objective='val_loss',
#     max_trials=10)

Evaluate on test data
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 95ms/step - accuracy: 0.7575 - loss: 0.6990
test loss, test acc: [0.698962390422821, 0.7574931979179382]


In [11]:
# tuner.search(train_ds, epochs=5, validation_data=val_ds)
# best_model = tuner.get_best_models()[0]

In [12]:
# best_model.predict(val_ds)